# OIBSIP Data Analytics — Task 3: Cleaning Data

**Objective:** Demonstrate professional data-cleaning skills by transforming a deliberately messy customer dataset into a clean, analysis-ready dataset and documenting every decision.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
DATA_DIR=Path('data')
raw=pd.read_csv(DATA_DIR/'messy_customer_data.csv')
raw.head()

## 1. Data Quality Report
The raw file intentionally contains missing values, duplicates, inconsistent formatting, invalid dates and numeric outliers.

In [ ]:
quality_report=pd.DataFrame({'dtype':raw.dtypes.astype(str),'null_count':raw.isna().sum(),'unique_values':raw.nunique(dropna=True)})
print('Shape:',raw.shape)
print('Duplicate rows:',raw.duplicated().sum())
quality_report

## 2. Missing Data Handling
Text fields use `Unknown`; numeric fields use median imputation; invalid dates are coerced and filled with the mode date. These choices are documented rather than silently changing values.

In [ ]:
clean=raw.copy()
clean['Customer_Name']=clean['Customer_Name'].fillna('Unknown')
clean['Gender']=clean['Gender'].astype('string').str.strip().str.lower().replace({'m':'Male','male':'Male','f':'Female','female':'Female'}).fillna('Unknown')
clean['City']=clean['City'].astype('string').str.strip().str.title().fillna('Unknown')
clean['Order_Date']=pd.to_datetime(clean['Order_Date'],errors='coerce')
clean['Age']=pd.to_numeric(clean['Age'],errors='coerce')
clean['Purchase_Amount']=pd.to_numeric(clean['Purchase_Amount'],errors='coerce')
clean['Age']=clean['Age'].fillna(clean['Age'].median())
clean['Purchase_Amount']=clean['Purchase_Amount'].fillna(clean['Purchase_Amount'].median())
clean['Order_Date']=clean['Order_Date'].fillna(clean['Order_Date'].mode()[0])

## 3. Duplicate Removal

In [ ]:
duplicate_count=clean.duplicated().sum()
clean=clean.drop_duplicates().copy()
print('Duplicates removed:',duplicate_count)

## 4. Standardisation
Gender and city values are normalised to consistent labels.

In [ ]:
clean[['Gender','City']].drop_duplicates().sort_values(['Gender','City']).head(20)

## 5. Outlier Detection with IQR
The IQR rule is used for numeric columns. Extreme values are capped at IQR fences instead of deleting rows, preserving customer records while limiting the influence of anomalies.

In [ ]:
def iqr_bounds(s):
    q1,q3=s.quantile([.25,.75]); iqr=q3-q1
    return q1-1.5*iqr,q3+1.5*iqr
for col in ['Age','Purchase_Amount']:
    lo,hi=iqr_bounds(clean[col]); count=((clean[col]<lo)|(clean[col]>hi)).sum()
    print(col,'outliers:',count,'bounds:',round(lo,2),round(hi,2))
    clean[col]=clean[col].clip(lo,hi)

## 6. Data Type Correction

In [ ]:
clean.dtypes

## 7. Before vs After Summary

In [ ]:
summary=pd.DataFrame({'Metric':['Rows','Duplicate rows','Missing cells','Order_Date dtype is datetime'],'Before':[len(raw),raw.duplicated().sum(),int(raw.isna().sum().sum()),False],'After':[len(clean),clean.duplicated().sum(),int(clean.isna().sum().sum()),pd.api.types.is_datetime64_any_dtype(clean['Order_Date'])]})
summary

## 8. Save Cleaned Dataset

In [ ]:
clean.to_csv(DATA_DIR/'cleaned_customer_data.csv',index=False)
summary.to_csv(DATA_DIR/'before_after_summary.csv',index=False)
print('Saved cleaned_customer_data.csv and before_after_summary.csv')

## Conclusion
The workflow converts inconsistent text, invalid dates, missing values, duplicate rows and numeric outliers into an analysis-ready dataset. The important principle is to document why each cleaning decision was made.